In [11]:
# import  os
# num_cores = "1"
# os.environ["OPENBLAS_NUM_THREADS"] = num_cores
# os.environ["OMP_NUM_THREADS"] = num_cores
# os.environ["MKL_NUM_THREADS"] = num_cores

In [12]:
import sys
sys.path.append('../')

import numpy as np
import scipy.sparse as ssp
import matplotlib.pyplot as plt
import qutip as qt
import scqubits as scq
from matplotlib.colors import LogNorm
from tqdm import tqdm
from qutip.qip.operations import rz, cz_gate
import cmath
from sympy import symbols
from scipy.sparse.linalg import eigsh
import utils_2Q_gate_zp as ut
from joblib import Parallel, delayed
from IPython.display import display, Math
# ut.set_fig_font() ### Set various sizes in plotting
import networkx as nx
from multiprocessing import Pool
import pandas as pd

In [13]:
truc1, truc_tot, charge_pick  = 300, 1000, True
normalize = True
folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
n_theta0_dress = pd.read_csv(folder+ 'n_theta0_dress.txt').to_numpy()
n_theta1_dress = pd.read_csv(folder+ 'n_theta1_dress.txt').to_numpy()
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()

In [14]:
truc_tot_2 = 500
truc_list = np.arange(truc_tot_2)
hspace_full = hspace_full[:truc_tot_2]
eval_tot = eval_tot[:truc_tot_2]

n_theta0_dress = ut.truncate_2(n_theta0_dress, truc_list)
n_theta1_dress = ut.truncate_2(n_theta1_dress, truc_list)
# print(np.shape(n_theta0_dress))
logi_state = ['0-0', '0-2', '2-0', '2-2']

# core_states = logi_state + ['4-5']
# idx_0 = hspace_full.index('0-2')
# idx_1 = hspace_full.index('2-2')
# idx_2 = hspace_full.index('4-5')

core_states = logi_state + ['8-2']
idx_0 = hspace_full.index('0-2')
idx_1 = hspace_full.index('2-2')
idx_2 = hspace_full.index('8-2')

W_0_2 = eval_tot[idx_2] - eval_tot[idx_0]
W_1_2 = eval_tot[idx_2] - eval_tot[idx_1]
print('E_45: W_0_2 = ', np.round(2*np.pi*W_0_2, 3), ', W_1_2 = ', np.round(2*np.pi*W_1_2, 3))


E_45: W_0_2 =  45.546 , W_1_2 =  23.753


### Truncation Estimate

In [15]:
drive_term = n_theta0_dress
max_n_ij = np.max(np.abs(drive_term.full()))
A = 0.02
######
# max_n_ij = np.max(drive_term) # Normalize based on n
population_rate = np.zeros((truc_tot, truc_tot), dtype=np.complex128)
population_rate_log = np.zeros((truc_tot, truc_tot), dtype=np.complex128)
rabi_df = []
G = nx.DiGraph()
for i, s_i in enumerate(hspace_full):
    for j, s_j in enumerate(hspace_full):
        if i < j:
            n_ij = np.abs(drive_term[i, j])
            if normalize:
                n_ij *= n_ij/max_n_ij
            # normalization =  n_ij/max_n_ij
            delta_1 = abs(W_0_2 - (eval_tot[j] - eval_tot[i]))
            delta_2 = abs(W_1_2 - (eval_tot[j] - eval_tot[i]))

            population_rate[i,j] = ((A*n_ij)**2) / ((A*n_ij)**2 + delta_1**2) + ((A*n_ij)**2) / ((A*n_ij)**2 + delta_2**2)
            if population_rate[i,j] > 0:
                population_rate_log[i, j] = -np.log(population_rate[i,j])
                G.add_edge(s_i, s_j, weight=population_rate_log[i, j]) # Construct the graph

    rabi_df.append({"order": i, "i": s_i})
rabi_df = pd.DataFrame(rabi_df)
rabi_df.index = rabi_df["i"]
print('shape(population_rate_log)=', np.shape(population_rate_log))
# rabi_df

shape(population_rate_log)= (1000, 1000)


In [16]:
def shortest_path_to_core(target):
    shortest_path = ""
    shortest_path_len = np.inf
    for source in core_states[:-1]:
        if nx.has_path(G, source, target):
            path = nx.shortest_path(G, source=source, target=target,
                                    weight="weight")
            path_len = nx.shortest_path_length(G, source=source, target=target,
                                                weight="weight")
        if path_len < shortest_path_len:
            shortest_path_len = path_len
            shortest_path = ",".join([str(x) for x in path])
    return target, (shortest_path_len, shortest_path)

cutoff = 2
def all_path_to_core(target):
    path_tot = []
    if target in core_states:
        weight_tot = 1
    else:
        weight_tot = 0
        for source in core_states:
            for path in nx.all_simple_paths(G, source, target, cutoff=cutoff):
                weight_tot += np.exp( - nx.path_weight(G, path,'weight') )
                path_tot.append(path)
    return target, (weight_tot, path_tot)


### All path

In [17]:
# Find all_path_to_core
pool = Pool(processes=150)
shortest_path = {}
rabi_df["path"] = ""
rabi_df["path_len"] = 1.0
for idx, path in tqdm(pool.imap_unordered(all_path_to_core, hspace_full),
                total=truc_tot):
    shortest_path[idx[0]] = path
    rabi_df.at[idx, "path"] = path[1]
    rabi_df.at[idx, "path_len"] = path[0]

df_all = rabi_df.sort_values("path_len", ascending=False)
print('all_path -- cutoff=', cutoff)
df_all.iloc[:30]

 50%|█████     | 500/1000 [00:01<00:01, 291.17it/s]


all_path -- cutoff= 2


,order,i,path,path_len
i,,,,
0-0,0,0-0,[],1.000000e+00+0.000000e+ 00j
2-2,13,2-2,[],1.000000e+00+0.000000e+ 00j
0-2,3,0-2,[],1.000000e+00+0.000000e+ 00j
2-0,4,2-0,[],1.000000e+00+0.000000e+ 00j
8-2,37,8-2,[],1.000000e+00+0.000000e+ 00j
12-2,64,12-2,"[[0-0, 0-1, 12-2], [0-0, 1-0, 12-2], [0-0, 0-5...",1.973147e-04+0.000000e+ 00j
4-9,63,4-9,"[[0-0, 0-1, 4-9], [0-0, 1-0, 4-9], [0-0, 0-5, ...",4.604333e-05+0.000000e+ 00j
1-2,11,1-2,"[[0-0, 1-2], [0-2, 1-2], [2-0, 1-2]]",4.369682e-05+0.000000e+ 00j
1-0,2,1-0,"[[0-0, 1-0]]",3.959675e-05+0.000000e+ 00j


In [ ]:
num_state = 100
df_all_truc = df_all.iloc[:num_state].sort_values("order", ascending=True)
print(f'state_all ({num_state}/{truc_tot_2}) :')
data = df_all_truc['i']
dim = 10
for i in range(0, len(data), dim):  # Step size of 10
    print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

index_all = [hspace_full.index(i) for i in df_all_truc['i']]
print(index_all)

state_all 100 :
'0-0', '0-1', '1-0', '0-2', '2-0', '4-0', '1-1', '0-5', '2-1', '5-0' ,
'1-2', '2-2', '8-0', '1-4', '4-1', '2-4', '9-0', '1-5', '5-1', '4-2' ,
'2-5', '1-8', '12-0', '5-2', '8-1', '4-4', '13-0', '15-0', '9-1', '8-2' ,
'5-4', '4-5', '0-21', '18-0', '9-2', '5-5', '4-8', '1-13', '20-0', '8-4' ,
'2-12', '15-1', '5-8', '4-9', '12-2', '9-4', '8-5', '25-0', '13-2', '26-0' ,
'1-18', '15-2', '5-9', '4-12', '18-1', '9-5', '5-12', '15-4', '18-2', '2-21' ,
'8-9', '1-25', '9-8', '12-5', '20-2', '5-16', '22-2', '13-5', '24-2', '15-5' ,
'9-9', '18-4', '12-8', '25-2', '26-2', '9-12', '12-9', '45-0', '18-5', '13-9' ,
'20-5', '22-5', '30-2', '25-4', '15-9', '33-2', '34-2', '35-2', '37-2', '25-5' ,
'26-5', '26-8', '24-9', '30-5', '44-2', '45-2', '50-2', '56-2', '45-5', '64-2' ,
[0, 1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 18, 19, 20, 21, 22, 24, 25, 26, 27, 30, 31, 32, 33, 36, 37, 38, 39, 44, 45, 46, 48, 50, 51, 52, 53, 56, 60, 62, 63, 64, 66, 68, 69, 70, 71, 72, 73, 75, 76, 79, 81, 

In [37]:
all_82_eli = [
'0-0', '0-2', '2-0', '8-2', '2-2', '12-2', '1-2', '1-0', '5-2', '4-9' ,
'5-0', '8-5', '22-2', '2-1', '9-2', '5-8', '0-1', '5-5', '34-2', '2-5' ,
'13-2', '8-0', '0-5', '1-4', '30-2', '20-5', '15-5', '4-2', '15-0', '15-2' ,
'1-1', '26-5', '8-9', '24-2', '26-2', '25-2', '9-0', '20-2', '4-5', '12-5' ,
'4-0', '5-1', '18-2', '12-0', '37-2', '9-4', '9-5', '1-5', '13-0', '35-2' ,
'13-9', '8-1', '20-0', '4-1', '9-1', '33-2', '18-5', '9-9', '12-8', '1-13' ,
'18-0', '9-8', '5-4', '1-8', '0-21', '5-16', '15-1', '5-12', '26-0', '25-4' ,
'1-25', '9-12', '22-5', '18-4', '4-4', '1-18', '64-2', '2-4', '5-9', '15-4' ,
'12-9', '13-5', '44-2', '45-5', '2-12', '4-12', '45-0', '18-1', '4-8', '8-4' ,
'15-9', '25-5', '24-9', '2-21', '50-2', '45-2', '12-1', '0-8', '26-8', '12-4' ,
'25-0', '18-8', '28-2', '1-9', '2-8', '35-0', '50-5', '20-4', '41-0', '46-2' ,
'39-0', '25-1', '0-4', '22-0', '33-0', '41-2', '22-9', '34-5', '8-18', '39-2' ,
'22-4', '59-2', '58-2', '60-2', '2-25', '56-2', '4-13', '54-2', '50-0', '30-5' ,
'13-1', '26-9', '20-9', '34-9', '15-8', '4-16', '4-21', '8-12', '2-30', '33-5' ,
'51-2', '35-4', '24-5', '59-0', '1-21', '0-25', '2-9', '56-0', '65-0', '41-1' ,
'35-5', '24-0', '20-1', '30-4', '18-9', '13-8', '65-2', '9-16', '46-5', '25-8' ,
'51-5', '2-26', '0-30', '28-1', '46-0', '9-13', '0-12', '0-24', '28-5', '34-0' ,
'22-8', '13-4', '33-12', '5-18', '4-24', '0-45', '41-5', '39-5', '34-1', '12-20' ,
'37-5', '24-1', '30-1', '12-12', '0-18', '22-1', '0-9', '37-0', '44-0', '13-12' ,
'69-0', '33-1', '30-0', '41-8', '44-5', '2-13', '1-42', '26-1', '4-18', '13-16' ,

]
all_82_zl = [
'0-0', '0-1', '1-0', '0-2', '2-0', '0-4', '4-0', '1-1', '0-5', '2-1' ,
'5-0', '1-2', '0-8', '2-2', '8-0', '1-4', '4-1', '2-4', '9-0', '1-5' ,
'5-1', '4-2', '0-12', '2-5', '1-8', '12-0', '5-2', '8-1', '4-4', '13-0' ,
'15-0', '2-8', '1-9', '9-1', '8-2', '5-4', '4-5', '0-18', '2-9', '0-21' ,
'18-0', '9-2', '12-1', '5-5', '0-24', '4-8', '1-13', '20-0', '8-4', '22-0' ,
'2-12', '13-1', '24-0', '0-25', '15-1', '5-8', '4-9', '12-2', '9-4', '8-5' ,
'25-0', '13-2', '26-0', '1-18', '15-2', '5-9', '4-12', '0-30', '18-1', '12-4' ,
'9-5', '1-21', '8-8', '20-1', '4-13', '22-1', '4-16', '30-0', '13-4', '5-12' ,
'15-4', '24-1', '33-0', '18-2', '34-0', '2-21', '8-9', '35-0', '1-25', '9-8' ,
'12-5', '37-0', '20-2', '5-16', '22-2', '13-5', '25-1', '26-1', '24-2', '15-5' ,
'2-25', '39-0', '8-12', '2-26', '9-9', '18-4', '28-1', '12-8', '41-0', '0-45' ,
'4-21', '13-8', '25-2', '20-4', '22-4', '4-24', '26-2', '5-18', '15-8', '30-1' ,
'9-12', '44-0', '12-9', '28-2', '45-0', '33-1', '18-5', '2-30', '46-0', '34-1' ,
'13-9', '9-13', '9-16', '20-5', '22-5', '50-0', '30-2', '25-4', '15-9', '8-18' ,
'24-5', '12-12', '33-2', '18-8', '34-2', '35-2', '13-12', '41-1', '37-2', '22-8' ,
'56-0', '25-5', '26-5', '30-4', '18-9', '39-2', '13-16', '59-0', '33-4', '28-5' ,
'41-2', '35-4', '20-9', '22-9', '25-8', '65-0', '26-8', '24-9', '30-5', '12-18' ,
'44-2', '33-5', '45-2', '34-5', '35-5', '46-2', '12-20', '69-0', '37-5', '26-9' ,
'50-2', '51-2', '39-5', '54-2', '41-5', '56-2', '58-2', '34-9', '18-20', '59-2' ,
'44-5', '45-5', '60-2', '41-8', '46-5', '64-2', '65-2', '33-12', '50-5', '51-5' ,
]
list1 = all_82_eli
list2 = all_82_zl
common_elements = [item for item in list1 if item in list2]
only_in_list1 = [item for item in list1 if item not in list2]
only_in_list2 = [item for item in list2 if item not in list1]
unique_elements = only_in_list1 + only_in_list2

print(f"_state_all ({num_state}/{truc_tot_2}): \nOnly in Eli:", len(only_in_list1), only_in_list1)
print("Only in ZL:", len(only_in_list2), only_in_list2)

_state_all(len=200): 
Only in Eli: 4 ['0-9', '2-13', '1-42', '4-18']
Only in ZL: 4 ['8-8', '33-4', '12-18', '18-20']


### Shortest path

In [43]:
# Find shortest path for each node
pool = Pool(processes=150)
shortest_path = {}
rabi_df["path"] = ""
rabi_df["path_len"] = 1.0
for idx, path in tqdm(pool.imap_unordered(shortest_path_to_core, hspace_full),
                total=truc_tot):
    shortest_path[idx[0]] = path
    rabi_df.at[idx, "path"] = path[1]
    rabi_df.at[idx, "path_len"] = np.exp(-path[0])

df_short = rabi_df.sort_values("path_len", ascending=False)
print('shortest_path:')
df_short.iloc[:30]

 50%|█████     | 500/1000 [00:06<00:06, 79.67it/s] 

shortest_path:


,order,i,path,path_len
i,,,,
8-2,37,8-2,"2-2,8-2",1.000000e+00-0.000000e+ 00j
0-0,0,0-0,0-0,1.000000e+00+0.000000e+ 00j
2-2,13,2-2,2-2,1.000000e+00+0.000000e+ 00j
0-2,3,0-2,0-2,1.000000e+00+0.000000e+ 00j
2-0,4,2-0,2-0,1.000000e+00+0.000000e+ 00j
12-2,64,12-2,"2-2,8-2,12-2",6.577152e-05-0.000000e+ 00j
1-2,11,1-2,"0-2,1-2",4.369682e-05-0.000000e+ 00j
1-0,2,1-0,"0-0,1-0",3.959675e-05-0.000000e+ 00j
5-2,27,5-2,"2-2,5-2",3.553933e-05-0.000000e+ 00j


In [48]:
num_state = 100
df_short_truc = df_short.iloc[:num_state].sort_values("order", ascending=True)
print('state_short :')
data = df_short_truc['i']
for i in range(0, len(data), dim):  # Step size of 10
    print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

index_all = [hspace_full.index(i) for i in df_short_truc['i']]
print(index_all)

state_short :
'0-0', '0-1', '1-0', '0-2', '2-0', '4-0', '1-1', '0-5', '2-1', '5-0' ,
'1-2', '0-8', '2-2', '8-0', '1-4', '4-1', '2-4', '9-0', '1-5', '5-1' ,
'4-2', '2-5', '1-8', '12-0', '5-2', '8-1', '4-4', '13-0', '15-0', '9-1' ,
'8-2', '5-4', '4-5', '0-21', '18-0', '9-2', '12-1', '5-5', '4-8', '1-13' ,
'20-0', '8-4', '2-12', '15-1', '5-8', '4-9', '12-2', '9-4', '8-5', '25-0' ,
'13-2', '26-0', '1-18', '15-2', '5-9', '4-12', '18-1', '9-5', '5-12', '15-4' ,
'18-2', '2-21', '8-9', '1-25', '9-8', '12-5', '20-2', '5-16', '22-2', '13-5' ,
'24-2', '15-5', '9-9', '18-4', '12-8', '25-2', '26-2', '9-12', '12-9', '45-0' ,
'18-5', '13-9', '20-5', '22-5', '30-2', '25-4', '15-9', '33-2', '34-2', '35-2' ,
'37-2', '25-5', '26-5', '30-5', '44-2', '45-2', '50-2', '56-2', '45-5', '64-2' ,
[0, 1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22, 24, 25, 26, 27, 30, 31, 32, 33, 36, 37, 38, 39, 44, 45, 46, 47, 48, 50, 51, 52, 53, 56, 60, 62, 63, 64, 66, 68, 69, 70, 71, 72, 73, 75, 76, 79,

In [50]:
short_82_eli = [
'0-0', '0-2', '2-2', '8-2', '2-0', '12-2', '1-2', '1-0', '5-2', '5-0' ,
'4-9', '8-5', '2-1', '22-2', '0-1', '9-2', '5-8', '2-5', '5-5', '34-2' ,
'8-0', '0-5', '13-2', '1-4', '30-2', '20-5', '4-2', '15-5', '15-0', '15-2' ,
'1-1', '8-9', '26-5', '24-2', '26-2', '9-0', '20-2', '25-2', '4-5', '4-0' ,
'5-1', '12-5', '12-0', '18-2', '9-5', '1-5', '13-0', '37-2', '35-2', '9-4' ,
'8-1', '20-0', '4-1', '9-1', '13-9', '33-2', '1-13', '18-0', '18-5', '9-9' ,
'12-8', '5-4', '0-21', '9-8', '1-8', '26-0', '5-12', '5-16', '15-1', '22-5' ,
'25-4', '1-25', '4-4', '5-9', '2-4', '18-4', '9-12', '15-4', '1-18', '64-2' ,
'45-0', '12-9', '13-5', '4-8', '44-2', '8-4', '45-5', '25-5', '2-12', '12-1' ,
'0-8', '50-2', '4-12', '45-2', '2-21', '18-1', '15-9', '25-0', '1-9', '2-8' ,
]
list1 = short_82_eli
list2 = df_short_truc['i']
common_elements = [item for item in list1 if item in list2]
only_in_list1 = [item for item in list1 if item not in list2]
only_in_list2 = [item for item in list2 if item not in list1]
unique_elements = only_in_list1 + only_in_list2

print(f"_state_short ({num_state}/{truc_tot_2}): \nOnly in Eli:", len(only_in_list1), only_in_list1)
print("Only in ZL:", len(only_in_list2), only_in_list2)


_state_short (100/500): 
Only in Eli: 2 ['1-9', '2-8']
Only in ZL: 2 ['30-5', '56-2']


In [26]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)

In [ ]:
hspace_index = [hspace_full.index(i) for i in logi_state]
print('index_state :')
dim = 10
data = hspace_full
for i in range(0, len(data), dim):  # Step size of 10
    print(i,", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

if charge_pick==False:
    thresh = 1e-2
    for s in hspace_index:
        for i in range(truc_tot):
            if np.abs(n_theta0_dress[s, i]) > thresh and i not in hspace_index:
                hspace_index.append(i)
    hspace_index.sort()
    print('threshold=', thresh)
    print('truncation_tot =', truc_tot, ', truncation_2nd_tot =', len(hspace_index))
    print('index_state after charge pick :')

    hspace_tot = np.array(hspace_full)[hspace_index]
    dim = 10
    data = hspace_tot
    for i in range(0, len(data), dim):  # Step size of 10
        print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

index_state :
0 '0-0', '0-1', '1-0', '0-2', '2-0', '0-4', '4-0', '1-1', '0-5', '2-1' ,
10 '5-0', '1-2', '0-8', '2-2', '8-0', '1-4', '4-1', '0-9', '2-4', '9-0' ,
20 '1-5', '5-1', '4-2', '0-12', '2-5', '1-8', '12-0', '5-2', '0-13', '0-16' ,
30 '8-1', '4-4', '13-0', '15-0', '2-8', '1-9', '9-1', '8-2', '5-4', '4-5' ,
40 '0-18', '2-9', '1-12', '0-20', '0-21', '18-0', '9-2', '12-1', '5-5', '0-24' ,
50 '4-8', '1-13', '20-0', '8-4', '1-16', '22-0', '2-12', '13-1', '24-0', '0-25' ,
60 '15-1', '0-26', '5-8', '4-9', '12-2', '2-13', '9-4', '2-16', '8-5', '25-0' ,
70 '13-2', '26-0', '1-18', '15-2', '0-28', '5-9', '4-12', '28-0', '0-30', '18-1' ,
80 '12-4', '9-5', '1-20', '1-21', '0-33', '8-8', '0-34', '0-35', '1-24', '2-18' ,
90 '0-36', '20-1', '4-13', '22-1', '4-16', '30-0', '13-4', '5-12', '15-4', '24-1' ,
100 '33-0', '2-20', '18-2', '34-0', '2-21', '8-9', '35-0', '1-25', '9-8', '1-26' ,
110 '0-39', '12-5', '2-24', '37-0', '5-13', '20-2', '5-16', '22-2', '13-5', '25-1' ,
120 '0-42', '26-1', '24-2

In [ ]:
if truc_tot < 30:
    nx.draw(G, pos=nx.shell_layout(G), with_labels=True)
print(G['0-0']['1-0'])
qt.Qobj(population_rate_log)

{'weight': (7.60517024528502-0j)}


Quantum object: dims = [[1000], [1000]], shape = (1000, 1000), type = oper, isherm = False
Qobj data =
[[ 0.         10.04754427  7.60517025 ... 42.40559133  0.
   0.        ]
 [ 0.          0.          0.         ...  0.         36.51149341
  45.17936135]
 [ 0.          0.          0.         ...  0.         37.47918684
  41.75901104]
 ...
 [ 0.          0.          0.         ...  0.         13.57799005
  15.07190484]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]

In [ ]:
print(shortest_path_to_core('1-0'))
print(all_path_to_core('0-4'))
print(all_path_to_core('0-1'))
target = '0-4'
source = '0-0'
paths = nx.all_simple_paths(G, source=source, target=target, cutoff=cutoff)
weight_tot = sum([np.exp(- nx.path_weight(G, path=path, weight='weight'))
                   for path in paths])
print(weight_tot)
for path in nx.all_simple_paths(G, source=source, target=target, cutoff=cutoff):
    print(path)

(1.210603091631031e-09+0j)
['0-0', '0-1', '0-4']
['0-0', '1-0', '0-4']


In [ ]:
cnot_80 = ['0-0', '0-1', '1-0', '0-2', '2-0', '4-0', '1-1', '0-5', '2-1', '5-0' ,
'1-2', '0-8', '2-2', '8-0', '1-4', '4-1', '0-9', '2-4', '9-0', '1-5' ,
'5-1', '4-2', '2-5', '1-8', '12-0', '5-2', '0-13', '8-1', '4-4', '13-0' ,
'15-0', '2-8', '1-9', '9-1', '8-2', '5-4', '4-5', '0-18', '1-12', '0-20' ,
'0-21', '18-0', '9-2', '12-1', '5-5', '4-8', '1-13', '20-0', '8-4', '22-0' ,
'13-1', '24-0', '15-1', '5-8', '4-9', '12-2', '2-13', '9-4', '8-5', '25-0' ,
'13-2', '26-0', '15-2', '5-9', '4-12', '28-0', '0-30', '18-1', '12-4', '9-5' ,
'1-20', '1-21', '8-8', '0-35', '0-36', '20-1', '4-13', '22-1', '30-0', '13-4' ,
'5-12', '15-4', '24-1', '33-0', '18-2', '34-0', '2-21', '8-9', '35-0', '1-25' ,
'9-8', '12-5', '37-0', '20-2', '22-2', '13-5', '25-1', '26-1', '24-2', '15-5' ,
'39-0', '9-9', '18-4', '28-1', '12-8', '41-0', '4-21', '13-8', '25-2', '8-13' ,
'20-4', '8-16', '22-4', '4-24', '26-2', '5-18', '15-8', '30-1', '9-12', '24-4' ,
'44-0', '12-9', '28-2', '45-0', '33-1', '18-5', '2-30', '46-0', '34-1', '35-1' ,
'5-21', '13-9', '9-13', '37-1', '20-5', '50-0', '51-0', '30-2', '25-4', '15-9' ,
'26-4', '24-5', '33-2', '39-1', '34-2', '35-2', '54-0', '28-4', '41-1', '37-2' ,
'22-8', '56-0', '26-5', '30-4', '58-0', '18-9', '39-2', '59-0', '33-4', '15-13' ,
'44-1', '34-4', '60-0', '45-1', '41-2', '35-4', '46-1', '64-0', '37-4', '65-0' ,
'26-8', '50-1', '51-1', '44-2', '33-5', '45-2', '34-5', '35-5', '69-0', '70-0' ,
'15-18', '50-2', '74-0', '75-0', '77-0', '33-8', '58-1', '34-8', '44-4', '41-5' ,
'60-1', '46-4', '81-0', '56-2', '65-1', '59-2', '60-2', '74-1', '98-0', '123-0' ,
]

[0, 1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22, 24, 25, 26, 27, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 55, 56, 57, 58, 59, 60, 62, 63, 64, 66, 68, 69, 70, 71, 72, 73, 75, 76, 78, 79, 80, 81, 82, 83, 85, 87, 90, 91, 92, 95, 96, 97, 98, 100, 102, 103, 104, 105, 106, 107, 108, 111, 115, 116, 117, 118, 121, 122, 123, 125, 126, 129, 130, 134, 135, 138, 139, 141, 142, 145, 147, 148, 149, 150, 153, 157, 158, 159, 160, 161, 162, 163, 164, 165, 168, 171, 176, 177, 183, 184, 186, 187, 189, 190, 191, 192, 195, 196, 197, 198, 201, 202, 203, 205, 213, 214, 215, 226, 230, 235, 241, 242, 248, 258, 259, 265, 268, 275, 280, 282, 283, 286, 293, 296, 297, 298, 301, 302, 306, 313, 320, 322, 331, 336, 339, 361, 373, 377, 398, 401, 418, 424, 431, 437, 440, 443, 444, 446, 463, 476, 486, 487, 549, 554, 578, 587, 598, 603, 611, 636, 651, 680, 922]


In [ ]:
aa = df_all_truc['i'].to_numpy().tolist()
index = [hspace_full.index(i) for i in aa]
index

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 72,
 73,
 75,
 76,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 94,
 97,
 98,
 99,
 101,
 102,
 104,
 105,
 107,
 108,
 109,
 110,
 111,
 112,
 114,
 116,
 117,
 120,
 123,
 125,
 127,
 128,
 129,
 133,
 136,
 138,
 140,
 143,
 147,
 148,
 150,
 151,
 153,
 156,
 157,
 158,
 160,
 164,
 165,
 166,
 167,
 170,
 172,
 173,
 175,
 177,
 178,
 179,
 181,
 182,
 183,
 185,
 187,
 193,
 199,
 204,
 208,
 209,
 210,
 227,
 236,
 243,
 254,
 257,
 260,
 263,
 265,
 269,
 271,
 274,
 276,
 278,
 284,
 285,
 290,
 295,
 303,
 304,
 310,
 324,
 327,
 329,
 332,
 333,
 348,
 364,
 366,
 379,
 387,
 391,
 392,
 401,
 402,
 403,
 406,
 409

In [ ]:
df_all_truc = df_all.iloc[:200].sort_values("order", ascending=True)
print('state_all :')
data = df_all_truc['i']
for i in range(0, len(data), dim):  # Step size of 10
    print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

state_all :
'0-0', '0-1', '1-0', '0-2', '2-0', '0-4', '4-0', '1-1', '0-5', '2-1' ,
'5-0', '1-2', '0-8', '2-2', '8-0', '1-4', '4-1', '0-9', '2-4', '9-0' ,
'1-5', '5-1', '4-2', '0-12', '2-5', '1-8', '12-0', '5-2', '0-13', '0-16' ,
'8-1', '4-4', '13-0', '15-0', '2-8', '1-9', '9-1', '8-2', '5-4', '4-5' ,
'0-18', '2-9', '1-12', '0-20', '0-21', '18-0', '9-2', '12-1', '5-5', '0-24' ,
'4-8', '1-13', '20-0', '8-4', '1-16', '22-0', '2-12', '13-1', '24-0', '0-25' ,
'15-1', '0-26', '5-8', '4-9', '12-2', '2-13', '9-4', '2-16', '8-5', '25-0' ,
'13-2', '1-18', '15-2', '5-9', '4-12', '0-30', '18-1', '12-4', '9-5', '1-20' ,
'1-21', '0-33', '8-8', '0-34', '0-35', '1-24', '2-18', '0-36', '4-16', '5-12' ,
'15-4', '24-1', '2-20', '18-2', '2-21', '8-9', '1-25', '9-8', '1-26', '0-39' ,
'12-5', '2-24', '5-13', '5-16', '22-2', '0-42', '15-5', '2-25', '8-12', '2-26' ,
'9-9', '0-44', '1-30', '0-45', '0-46', '1-33', '22-4', '4-24', '5-18', '1-34' ,
'15-8', '2-28', '9-12', '24-4', '12-9', '18-5', '2-30', '0-52', '

In [ ]:
print('cutoff=', cutoff)
target = '5-2'
print(shortest_path_to_core(target))
idx, path = all_path_to_core(target)
for i in path[1]:
    print(i)


cutoff= 2
('5-2', ((1.1144374429708779+0j), '2-2,5-2'))
['0-0', '5-2']
['0-2', '5-2']
['2-0', '5-2']
['2-2', '5-2']
['5-0', '2-2', '5-2']
['5-0', '0-9', '5-2']
['5-0', '2-4', '5-2']
['5-0', '9-0', '5-2']
['5-0', '1-5', '5-2']
['5-0', '5-1', '5-2']
['5-0', '4-2', '5-2']
['5-0', '0-12', '5-2']
['5-0', '1-8', '5-2']
['5-0', '12-0', '5-2']


In [ ]:
print('cutoff=', cutoff)
target = '5-2'
print(shortest_path_to_core(target))
idx, path = all_path_to_core(target)
for i in path[1]:
    print(i)


cutoff= 2
('5-2', ((1.1144374429708779+0j), '2-2,5-2'))
['0-0', '5-2']
['0-2', '5-2']
['2-0', '5-2']
['2-2', '5-2']
['5-0', '2-2', '5-2']
['5-0', '0-9', '5-2']
['5-0', '2-4', '5-2']
['5-0', '9-0', '5-2']
['5-0', '1-5', '5-2']
['5-0', '5-1', '5-2']
['5-0', '4-2', '5-2']
['5-0', '0-12', '5-2']
['5-0', '1-8', '5-2']
['5-0', '12-0', '5-2']


In [ ]:
# Find shortest path for each node
pool = Pool(processes=150)
shortest_path = {}
rabi_df["path"] = ""
rabi_df["path_len"] = 0.0
rabi_df["degree"] = -1
for idx, path in tqdm(pool.imap_unordered(shortest_path_to_core, hspace_full),
                total=truc_tot):
    shortest_path[idx[0]] = path
    rabi_df.at[idx, "path"] = path[1]
    rabi_df.at[idx, "path_len"] = np.exp(-path[0])
    rabi_df.at[idx, "degree"] = path[1].count(",")

# print(population_rate[index_state.index('0-0'), index_state.index('0-1')] *
#       population_rate[index_state.index('0-1'), index_state.index('0-4')])
# print(population_rate[index_state.index('0-0'), index_state.index('1-0')] *
#       population_rate[index_state.index('1-0'), index_state.index('0-4')])
print(rabi_df.sort_values("path_len", ascending=False).iloc[:15])
# qt.Qobj(population_rate[:10,:10])

100%|██████████| 1000/1000 [00:03<00:00, 258.22it/s]

     order    i         path            path_len  degree
i                                                       
0-0      0  0-0          0-0  1.000000+0.000000j       0
2-2     13  2-2          2-2  1.000000+0.000000j       0
0-2      3  0-2          0-2  1.000000+0.000000j       0
2-0      4  2-0          2-0  1.000000+0.000000j       0
5-0     10  5-0      2-0,5-0  1.000000-0.000000j       1
5-2     27  5-2      2-2,5-2  0.328100-0.000000j       1
5-1     21  5-1  2-0,5-0,5-1  0.063630-0.000000j       2
0-1      1  0-1      0-0,0-1  0.033558-0.000000j       1
2-1      9  2-1      2-0,2-1  0.027706-0.000000j       1
1-0      2  1-0      0-0,1-0  0.022540-0.000000j       1
0-5      8  0-5      0-2,0-5  0.014840-0.000000j       1
2-5     24  2-5      2-2,2-5  0.013964-0.000000j       1
1-2     11  1-2      0-2,1-2  0.013392-0.000000j       1
4-0      6  4-0  0-0,0-1,4-0  0.007998-0.000000j       2
5-5     48  5-5  2-2,5-2,5-5  0.005663-0.000000j       2
